In [17]:
import pandas as pd

df = pd.read_csv("../models/hog_pca.csv")

X = df.drop("label", axis=1)
y = df["label"]

print(df.head())
print(df.shape)
print(y.value_counts())

          0         1         2         3         4         5         6  \
0  0.202303  0.501626  1.186199 -0.371371 -0.136793 -0.076230 -0.407002   
1  1.953665 -0.459767 -0.095813  0.723011 -0.357205  0.048984 -0.545812   
2 -0.330285  0.010939  2.515379  0.408827 -0.262966  0.678323 -0.456933   
3 -0.499286  1.047266  0.780613 -0.732237  0.215755  1.138388  0.282930   
4  0.324365  0.173774 -0.120909  1.110068 -0.578559  0.778288  0.294386   

          7         8         9  ...       191       192       193       194  \
0  0.833590 -0.042453 -0.106293  ...  0.099312  0.054471  0.075806  0.178201   
1  0.091803  0.508039 -0.128445  ...  0.014885 -0.037539  0.077711  0.215540   
2 -0.109575 -0.267256 -0.031114  ... -0.133839 -0.039047  0.102327  0.054518   
3 -0.264202  0.321746  0.258287  ...  0.094813  0.080843  0.113123  0.027981   
4 -0.745796 -0.072325  0.292275  ...  0.031504 -0.103231 -0.106877 -0.081406   

        195       196       197       198       199  label  
0 -0.11

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train shape: ", X_train.shape)
print("Test shape: ", X_test.shape)
print("Distribucija klasa u train skupu:")
print(y_train.value_counts())
print("\nDistribucija klasa u test skupu:")
print(y_test.value_counts())

Train shape:  (968, 200)
Test shape:  (415, 200)
Distribucija klasa u train skupu:
label
2    328
1    324
0    316
Name: count, dtype: int64

Distribucija klasa u test skupu:
label
2    140
1    139
0    136
Name: count, dtype: int64


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    "SVM (RBF Kernel)": SVC(kernel="rbf", probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)
}

models

{'SVM (RBF Kernel)': SVC(probability=True, random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
 'Gradient Boosting': GradientBoostingClassifier(random_state=42),
 'Neural Network (MLP)': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42),
 'KNN (k=5)': KNeighborsClassifier()}

In [20]:
from sklearn.metrics import accuracy_score

trained_models = {}
predictions = {}

for name, model in models.items():
    print(f"Treniram model: {name}")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    trained_models[name] = model
    predictions[name] = y_pred

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}\n")

Treniram model: SVM (RBF Kernel)
Accuracy: 0.5759

Treniram model: Random Forest
Accuracy: 0.5253

Treniram model: Gradient Boosting
Accuracy: 0.5446

Treniram model: Neural Network (MLP)
Accuracy: 0.5301

Treniram model: KNN (k=5)
Accuracy: 0.4699



In [21]:
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import pandas as pd

results = []

for name, y_pred in predictions.items():
    print("="*70)
    print(f"METRIKE ZA MODEL: {name}")
    print("="*70)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confussion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    macro_precision = precision_score(y_test, y_pred, average="macro")
    macro_recall = recall_score(y_test, y_pred, average="macro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    micro_precision = precision_score(y_test, y_pred, average="micro")
    micro_recall = recall_score(y_test, y_pred, average="micro")
    micro_f1 = f1_score(y_test, y_pred, average="micro")

    results.append({
        "Model": name,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Micro Precision": micro_precision,
        "Micro Recall": micro_recall,
        "Micro F1": micro_f1
    })

results_df = pd.DataFrame(results)
results_df

METRIKE ZA MODEL: SVM (RBF Kernel)

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.65      0.64       136
           1       0.46      0.35      0.40       139
           2       0.60      0.72      0.66       140

    accuracy                           0.58       415
   macro avg       0.56      0.58      0.57       415
weighted avg       0.56      0.58      0.57       415

Confussion Matrix:
[[ 89  33  14]
 [ 38  49  52]
 [ 14  25 101]]
METRIKE ZA MODEL: Random Forest

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.57      0.57       136
           1       0.42      0.32      0.36       139
           2       0.55      0.69      0.62       140

    accuracy                           0.53       415
   macro avg       0.51      0.53      0.52       415
weighted avg       0.51      0.53      0.51       415

Confussion Matrix:
[[77 33 26]
 [43 44 52]
 [14 29 97]]
ME

,Model,Macro Precision,Macro Recall,Macro F1,Micro Precision,Micro Recall,Micro F1
0,SVM (RBF Kernel),0.564647,0.576119,0.566318,0.575904,0.575904,0.575904
1,Random Forest,0.514669,0.525193,0.515142,0.525301,0.525301,0.525301
2,Gradient Boosting,0.541549,0.544185,0.538068,0.544578,0.544578,0.544578
3,Neural Network (MLP),0.530792,0.530089,0.530423,0.530120,0.530120,0.530120
4,KNN (k=5),0.491740,0.467999,0.445527,0.469880,0.469880,0.469880


In [22]:
from sklearn.model_selection import GridSearchCV

param_grid_svm = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01, 0.001],
    "kernel": ["rbf"]
}

grid_svm = GridSearchCV(
    SVC(probability=True),
    param_grid_svm,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Najbolji parametri: ", grid_svm.best_params_)
print("Najbolji F1 (CV): ", grid_svm.best_score_)

Najbolji parametri:  {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Najbolji F1 (CV):  0.5525386327841916


In [23]:
param_grid_rf = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("Najbolji parametri: ", grid_rf.best_params_)
print("Najbolji F1 (CV): ", grid_rf.best_score_)

Najbolji parametri:  {'max_depth': 20, 'min_samples_split': 10, 'n_estimators': 400}
Najbolji F1 (CV):  0.5202947729537977


In [24]:
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.01, 0.05, 0.1],
    "max_depth": [2, 3, 4]
}

grid_gb = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_gb,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)

print("Najbolji parametri:", grid_gb.best_params_)
print("Najbolji F1 (CV):", grid_gb.best_score_)

Najbolji parametri: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Najbolji F1 (CV): 0.5283409637244747


In [25]:
param_grid_mlp = {
    "hidden_layer_sizes": [(128,64), (256,128), (64,32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01],
    "max_iter": [500]
}

grid_mlp = GridSearchCV(
    MLPClassifier(random_state=42),
    param_grid_mlp,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_mlp.fit(X_train, y_train)

print("Najbolji parametri:", grid_mlp.best_params_)
print("Najbolji F1 (CV):", grid_mlp.best_score_)


Najbolji parametri: {'activation': 'relu', 'hidden_layer_sizes': (256, 128), 'learning_rate_init': 0.001, 'max_iter': 500}
Najbolji F1 (CV): 0.5163314461000305


In [26]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "p": [1, 2]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Najbolji parametri:", grid_knn.best_params_)
print("Najbolji F1 (CV):", grid_knn.best_score_)


Najbolji parametri: {'n_neighbors': 3, 'p': 2, 'weights': 'uniform'}
Najbolji F1 (CV): 0.44003040347539574


In [27]:
best_models = {
    "SVM (RBF kernel)": SVC(
        C=grid_svm.best_params_["C"],
        gamma=grid_svm.best_params_["gamma"],
        kernel="rbf",
        probability=True,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=grid_rf.best_params_["n_estimators"],
        max_depth=grid_rf.best_params_["max_depth"],
        min_samples_split=grid_rf.best_params_["min_samples_split"],
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=grid_gb.best_params_["n_estimators"],
        learning_rate=grid_gb.best_params_["learning_rate"],
        max_depth=grid_gb.best_params_["max_depth"],
        random_state=42
    ),

    "Neural Network (MLP)": MLPClassifier(
        hidden_layer_sizes=grid_mlp.best_params_["hidden_layer_sizes"],
        activation=grid_mlp.best_params_["activation"],
        learning_rate_init=grid_mlp.best_params_["learning_rate_init"],
        max_iter=grid_mlp.best_params_["max_iter"],
        random_state=42
    ),

    "KNN (k=3)": KNeighborsClassifier(
        n_neighbors=grid_knn.best_params_["n_neighbors"],
        weights=grid_knn.best_params_["weights"],
        p=grid_knn.best_params_["p"]
    )
}

best_predictions = {}

print("Ponovno treniramo modele s najboljim parametrima...\n")

for name, model in best_models.items():
    print(f"Treniram: {name}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    best_predictions[name] = y_pred


Ponovno treniramo modele s najboljim parametrima...

Treniram: SVM (RBF kernel)
Treniram: Random Forest
Treniram: Gradient Boosting
Treniram: Neural Network (MLP)
Treniram: KNN (k=3)


In [28]:
final_results = []

for name, y_pred in best_predictions.items():
    print("="*70)
    print(f"FINALNE METRIKE ZA OPTIMIZIRANI MODEL: {name}")
    print("="*70)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    macro_precision = precision_score(y_test, y_pred, average="macro")
    macro_recall = recall_score(y_test, y_pred, average="macro")
    macro_f1 = f1_score(y_test, y_pred, average="macro")

    micro_precision = precision_score(y_test, y_pred, average="micro")
    micro_recall = recall_score(y_test, y_pred, average="micro")
    micro_f1 = f1_score(y_test, y_pred, average="micro")

    final_results.append({
        "Model": name,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Micro Precision": micro_precision,
        "Micro Recall": micro_recall,
        "Micro F1": micro_f1
    })

final_results_df = pd.DataFrame(final_results)
final_results_df


FINALNE METRIKE ZA OPTIMIZIRANI MODEL: SVM (RBF kernel)

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.65      0.64       136
           1       0.46      0.35      0.40       139
           2       0.60      0.72      0.66       140

    accuracy                           0.58       415
   macro avg       0.56      0.58      0.57       415
weighted avg       0.56      0.58      0.57       415

Confusion Matrix:
[[ 89  33  14]
 [ 38  49  52]
 [ 14  25 101]]
FINALNE METRIKE ZA OPTIMIZIRANI MODEL: Random Forest

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.60      0.60       136
           1       0.38      0.24      0.30       139
           2       0.55      0.74      0.63       140

    accuracy                           0.53       415
   macro avg       0.51      0.53      0.51       415
weighted avg       0.51      0.53      0.51       415

Confusion Matrix:

,Model,Macro Precision,Macro Recall,Macro F1,Micro Precision,Micro Recall,Micro F1
0,SVM (RBF kernel),0.564647,0.576119,0.566318,0.575904,0.575904,0.575904
1,Random Forest,0.510777,0.530134,0.510497,0.530120,0.530120,0.530120
2,Gradient Boosting,0.541549,0.544185,0.538068,0.544578,0.544578,0.544578
3,Neural Network (MLP),0.533304,0.532296,0.531481,0.532530,0.532530,0.532530
4,KNN (k=3),0.477229,0.468385,0.449392,0.469880,0.469880,0.469880


In [29]:
import joblib

best_model = best_models["SVM (RBF kernel)"]

joblib.dump(best_model, "../models/best_hog_lbp_model.pkl")

['../models/best_hog_lbp_model.pkl']